# Prepare the trained Rāmāyaṇa model for testing and deployment
Click Runtime → Run all and approve Drive access. This verifies the full checkpoint checksum, then writes a separate weights-only copy without changing your resumable checkpoints. Tell your assistant when it prints Ready.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import torch

drive.mount('/content/drive')
folder = Path('/content/drive/MyDrive/Sanskrit-LLM/ramayana-v1')
checkpoints = sorted(p for p in folder.glob('tokens-*.pt')
                     if Path(str(p) + '.sha256').is_file())
if not checkpoints:
    raise RuntimeError('No complete checkpoint found in ' + str(folder))
checkpoint = checkpoints[-1]
h = hashlib.sha256()
with checkpoint.open('rb') as f:
    for block in iter(lambda: f.read(1024 * 1024), b''):
        h.update(block)
expected = Path(str(checkpoint) + '.sha256').read_text().split()[0]
if h.hexdigest() != expected:
    raise RuntimeError('Checkpoint checksum mismatch: ' + checkpoint.name)
data = torch.load(checkpoint, map_location='cpu', weights_only=False)
if data['metadata'].get('kind') != 'ramayana_v1':
    raise RuntimeError('Not a Ramayana training checkpoint')
out = folder.parent / 'ramayana-deployment-weights.pt'
tmp = folder.parent / 'ramayana-deployment-weights.pt.tmp'
torch.save({'config': data['config'], 'state_dict': data['state_dict'],
            'metadata': data['metadata']}, tmp)
tmp.replace(out)
print('Ready:', out, 'from', checkpoint.name, flush=True)
print('Additional positions:', data['metadata']['additional_nonpadding_positions'], flush=True)
print('Approximate total positions:', data['metadata']['approx_total_positions'], flush=True)
print('Source checkpoint SHA256:', expected, flush=True)
